In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("/MedicalRecordsDataset.csv", parse_dates=["Test_Date"])
df.head()

,Name,Age,Sex,Zip_Code,Disease,Test_Date,Result
0,Andrew Robinson,57,Male,64217,Hypertension,2021-10-31,Negative
1,Andrew Robinson,57,Male,64219,Asthma,2021-03-15,Positive
2,Andrew Robinson,57,Male,64221,COVID-19,2022-06-02,Positive
3,Andrew Robinson,57,Male,64221,Cancer,2023-01-23,Positive
4,Roy Rodriguez,79,Male,24012,Diabetes,2020-01-12,Positive


In [3]:
# Checking how many distinct people there are, how many tests each has, the different age groups, which diseases and ZIP codes exist, and the date range.
# We get to know our data here. The dataset contains 414 test records for 100 individuals (2 to 6 tests each) between October 2019 and October 2024. Cancer accounts for 38% of records.

print(df.shape)
print("People:", df["Name"].nunique())
print("\n")
print(df.groupby("Name").size().describe())
print("\n")
print("Age range:", df["Age"].min(), "to", df["Age"].max())
print(df["Age"].describe())
print("\n")
print(df["Sex"].value_counts())
print("\n")
print("Distinct ZIP codes:", df["Zip_Code"].nunique())
print("ZIP range:", df["Zip_Code"].min(), "to", df["Zip_Code"].max())
print("\n")
print(df["Disease"].value_counts())
print("\n")
print(df["Result"].value_counts())
print("\n")
print("Dates:", df["Test_Date"].min(), "to", df["Test_Date"].max())

(414, 7)
People: 100


count    100.000000
mean       4.140000
std        1.470381
min        2.000000
25%        3.000000
50%        4.000000
75%        6.000000
max        6.000000
dtype: float64


Age range: 20 to 80
count    414.000000
mean      47.147343
std       18.891176
min       20.000000
25%       30.000000
50%       47.000000
75%       62.000000
max       80.000000
Name: Age, dtype: float64


Sex
Male      218
Female    196
Name: count, dtype: int64


Distinct ZIP codes: 332
ZIP range: 10033 to 99411


Disease
Cancer           159
Hypertension      55
Asthma            53
Heart Disease     52
Diabetes          48
COVID-19          47
Name: count, dtype: int64


Result
Negative    218
Positive    196
Name: count, dtype: int64


Dates: 2019-10-21 00:00:00 to 2024-10-11 00:00:00


In [4]:
# 99 of 100 individuals have different ZIP codes across their records, so ZIP was treated as a high risk quasi-identifier and generalized.

zips_per_person = df.groupby("Name")["Zip_Code"].nunique()
print((zips_per_person > 1).sum(), "of", len(zips_per_person), "people have more than one ZIP")

99 of 100 people have more than one ZIP


In [5]:
# We classify the columns
import hmac, hashlib

DIRECT_ID = ["Name"]
QI_ID     = ["Age", "Sex", "Zip_Code", "Test_Date"]
SENSITIVE = ["Disease", "Result"]

In [6]:
# Parameters to make things tunable

SECRET_KEY = b"hello"
AGE_WIDTH  = 20
K_TARGET   = 5

In [7]:
# Generalisation functions

def pseudonym(name):
    return "P-" + hmac.new(SECRET_KEY, name.encode(), hashlib.sha256).hexdigest()[:8]  # replaces a name with a code made by a keyed hash

def gen_age(a, w=AGE_WIDTH):
    lo = (a // w) * w
    return lo.astype(str) + "-" + (lo + w - 1).astype(str)   # turns exact ages into ranges


def gen_region(zip_code):
    first_digit = zip_code // 10000
    return pd.cut(first_digit, [0, 3, 6, 9],
                  labels=["ZIP 1-3", "ZIP 4-6", "ZIP 7-9"]).astype(str) # keeps only the first ZIP digit, grouped into 3 regions

def gen_period(dates):
    return pd.cut(dates.dt.year, [2018, 2021, 2024],
                  labels=["2019-2021", "2022-2024"]).astype(str) # turns exact dates into 2 periods

In [8]:
# We add the pseudonym, generalize the QIs, and drop the original Name, ZIP and date columns.

anon = df.copy()
anon["Person"] = anon["Name"].apply(pseudonym)
anon["Age"]    = gen_age(anon["Age"])
anon["Region"] = gen_region(anon["Zip_Code"])
anon["Period"] = gen_period(anon["Test_Date"])
anon = anon.drop(columns=["Name", "Zip_Code", "Test_Date"])

QI_ANON = ["Age", "Sex", "Region", "Period"]
anon.head()

,Age,Sex,Disease,Result,Person,Region,Period
0,40-59,Male,Hypertension,Negative,P-671c6390,ZIP 4-6,2019-2021
1,40-59,Male,Asthma,Positive,P-671c6390,ZIP 4-6,2019-2021
2,40-59,Male,COVID-19,Positive,P-671c6390,ZIP 4-6,2022-2024
3,40-59,Male,Cancer,Positive,P-671c6390,ZIP 4-6,2022-2024
4,60-79,Male,Diabetes,Positive,P-adc95a7e,ZIP 1-3,2019-2021


In [9]:
# privacy measures

def k_anonymity(d, qi):
    return d.groupby(qi).size().min() # groups rows by the QIs and returns the smallest class size

def k_people(d, qi):
    return d.groupby(qi)["Person"].nunique().min() # does the same but counts distinct people, not rows

def l_distinct(d, qi, sa="Disease"):
    return d.groupby(qi)[sa].nunique().min() # gives the smallest number of different diseases in a class

def l_entropy(d, qi, sa="Disease"):
    def eff(s):
        p = s.value_counts(normalize=True)
        return np.exp(-(p * np.log(p)).sum())
    return d.groupby(qi)[sa].apply(eff).min() # computes entropy per class and takes the minimum

In [10]:
# Measure, then suppress

print("BEFORE suppression")
print("k =", k_anonymity(anon, QI_ANON), "| l =", l_distinct(anon, QI_ANON))

class_size = anon.groupby(QI_ANON)["Disease"].transform("size")
final = anon[class_size >= K_TARGET].copy()

print("\nAFTER suppression")
print("Rows suppressed: %.1f%%" % (100 * (1 - len(final) / len(anon))))
print("k (rows)            =", k_anonymity(final, QI_ANON))
print("k (distinct people) =", k_people(final, QI_ANON))
print("l distinct          =", l_distinct(final, QI_ANON))
print("l entropy           = %.2f" % l_entropy(final, QI_ANON))

BEFORE suppression
k = 1 | l = 1

AFTER suppression
Rows suppressed: 2.2%
k (rows)            = 5
k (distinct people) = 1
l distinct          = 3
l entropy           = 2.38


In [11]:
# we show each class with its disease counts, then save the anonymized file

final.groupby(QI_ANON)["Disease"].agg(lambda s: dict(s.value_counts()))
final.to_csv("/anonymized_medical_records.csv", index=False)

# Step 2: Utility evaluation (original vs anonymized)

In [12]:
# The original data is generalized in the same way (age bands, regions, periods) so both datasets can be compared band by band.
orig = df.copy()
orig["AgeBand"] = gen_age(orig["Age"])
orig["Region"]  = gen_region(orig["Zip_Code"])
orig["Period"]  = gen_period(orig["Test_Date"])
orig["Year"]    = orig["Test_Date"].dt.year

def pct(s): return (s * 100).round(1)

def tv(p, q):   # total variation distance between two distributions (0 = identical, 1 = completely different)
    idx = p.index.union(q.index)
    return 0.5 * (p.reindex(idx, fill_value=0) - q.reindex(idx, fill_value=0)).abs().sum()

In [13]:
# (a) Demographic distribution: age bands and sex
age_o = orig["AgeBand"].value_counts(normalize=True).sort_index()
age_a = final["Age"].value_counts(normalize=True).sort_index()
sex_o = orig["Sex"].value_counts(normalize=True)
sex_a = final["Sex"].value_counts(normalize=True)

print(pct(pd.concat([age_o, age_a], axis=1, keys=["original %", "anonymized %"]).fillna(0)))
print()
print(pct(pd.concat([sex_o, sex_a], axis=1, keys=["original %", "anonymized %"])))
print("\nTotal variation distance, age bands:", round(tv(age_o, age_a), 4))

       original %  anonymized %
20-39        44.0          44.9
40-59        27.5          26.2
60-79        27.1          27.7
80-99         1.4           1.2

        original %  anonymized %
Sex                             
Male          52.7          52.8
Female        47.3          47.2

Total variation distance, age bands: 0.0158


In [14]:
# (b) Disease prevalence over time (2 periods in the anonymized data)
b_o = pd.crosstab(orig["Period"], orig["Disease"])
b_a = pd.crosstab(final["Period"], final["Disease"])
print(b_o); print(); print(b_a)
print("\nShare of each disease per period (%), original:")
print(pct(b_o.div(b_o.sum(axis=1), axis=0)))
print("\nShare of each disease per period (%), anonymized:")
print(pct(b_a.div(b_a.sum(axis=1), axis=0)))
print("\nDetail lost by generalizing dates (original, per year):")
print(pd.crosstab(orig["Year"], orig["Disease"]))

Disease    Asthma  COVID-19  Cancer  Diabetes  Heart Disease  Hypertension
Period                                                                    
2019-2021      25        18      66        21             25            32
2022-2024      28        29      93        27             27            23

Disease    Asthma  COVID-19  Cancer  Diabetes  Heart Disease  Hypertension
Period                                                                    
2019-2021      24        18      63        21             24            32
2022-2024      26        29      91        27             27            23

Share of each disease per period (%), original:
Disease    Asthma  COVID-19  Cancer  Diabetes  Heart Disease  Hypertension
Period                                                                    
2019-2021    13.4       9.6    35.3      11.2           13.4          17.1
2022-2024    12.3      12.8    41.0      11.9           11.9          10.1

Share of each disease per period (%), anonymized:

In [15]:
# (c) Testing frequency per disease
c = pd.concat([orig["Disease"].value_counts(), final["Disease"].value_counts()],
              axis=1, keys=["original", "anonymized"])
c["lost"] = c["original"] - c["anonymized"]
c["orig share %"] = pct(c["original"] / c["original"].sum())
c["anon share %"] = pct(c["anonymized"] / c["anonymized"].sum())
print(c)
print("\nAverage tests per person:", round(orig.groupby("Name").size().mean(), 2),
      "(original) vs", round(final.groupby("Person").size().mean(), 2), "(anonymized)")

               original  anonymized  lost  orig share %  anon share %
Disease                                                              
Cancer              159         154     5          38.4          38.0
Hypertension         55          55     0          13.3          13.6
Asthma               53          50     3          12.8          12.3
Heart Disease        52          51     1          12.6          12.6
Diabetes             48          48     0          11.6          11.9
COVID-19             47          47     0          11.4          11.6

Average tests per person: 4.14 (original) vs 4.05 (anonymized)


In [16]:
# (d) Geographic distribution of diseases (by region)
d_o = pd.crosstab(orig["Region"], orig["Disease"])
d_a = pd.crosstab(final["Region"], final["Disease"])
sh_o = d_o.div(d_o.sum(axis=1), axis=0)
sh_a = d_a.div(d_a.sum(axis=1), axis=0)
print(d_o); print(); print(d_a)
print("\nLargest difference in a disease share between datasets (percentage points):",
      round((sh_o - sh_a).abs().max().max() * 100, 2))
print("Distinct ZIP codes in original:", df["Zip_Code"].nunique(), "| regions after generalization:", final["Region"].nunique())

Disease  Asthma  COVID-19  Cancer  Diabetes  Heart Disease  Hypertension
Region                                                                  
ZIP 1-3      18        18      56        20             17            21
ZIP 4-6      18        17      44        15             16            18
ZIP 7-9      17        12      59        13             19            16

Disease  Asthma  COVID-19  Cancer  Diabetes  Heart Disease  Hypertension
Region                                                                  
ZIP 1-3      16        18      54        20             17            21
ZIP 4-6      17        17      44        15             16            18
ZIP 7-9      17        12      56        13             18            16

Largest difference in a disease share between datasets (percentage points): 1.04
Distinct ZIP codes in original: 332 | regions after generalization: 3


In [17]:
# (e) Impact of age on disease testing
def mid(s):   # midpoint of a band such as "40-59"
    return s.str.split("-").apply(lambda x: (int(x[0]) + int(x[1])) / 2)

final["AgeMid"] = mid(final["Age"])
print(pd.concat([orig.groupby("Disease")["Age"].mean(), final.groupby("Disease")["AgeMid"].mean()],
                axis=1, keys=["exact age (original)", "band midpoint (anonymized)"]).round(2))
print("\nDisease share per age band (%), original:")
print(pct(pd.crosstab(orig["AgeBand"], orig["Disease"], normalize="index")))
print("\nDisease share per age band (%), anonymized:")
print(pct(pd.crosstab(final["Age"], final["Disease"], normalize="index")))
print("\nCorrelation age vs cancer test: original", round(np.corrcoef(orig["Age"], orig["Disease"] == "Cancer")[0, 1], 3),
      "| anonymized", round(np.corrcoef(final["AgeMid"], final["Disease"] == "Cancer")[0, 1], 3))

               exact age (original)  band midpoint (anonymized)
Disease                                                        
Asthma                        46.53                       45.10
COVID-19                      48.72                       47.80
Cancer                        47.03                       46.51
Diabetes                      48.04                       47.42
Heart Disease                 46.23                       45.97
Hypertension                  46.84                       46.59

Disease share per age band (%), original:
Disease  Asthma  COVID-19  Cancer  Diabetes  Heart Disease  Hypertension
AgeBand                                                                 
20-39      13.2      10.4    38.5      11.5           13.7          12.6
40-59      13.2      12.3    38.6      10.5           10.5          14.9
60-79      11.6      11.6    38.4      12.5           12.5          13.4
80-99      16.7      16.7    33.3      16.7           16.7           0.0

Diseas

In [18]:
# (f) Disease comorbidity: number of people who had a test for both diseases
def comorbidity(d, pid):
    m = pd.crosstab(d[pid], d["Disease"]).clip(upper=1)   # 1 if the person had at least one test for this disease
    return m.T.dot(m)

co_o = comorbidity(orig, "Name")
co_a = comorbidity(final, "Person")
print(co_o); print(); print(co_a)

iu = np.triu_indices(len(co_o), 1)
pairs_o = co_o.values[iu]
pairs_a = co_a.reindex(index=co_o.index, columns=co_o.columns).values[iu]
print("\nCorrelation of the pair counts:", round(np.corrcoef(pairs_o, pairs_a)[0, 1], 3))
print("Mean absolute difference per pair:", round(np.abs(pairs_o - pairs_a).mean(), 2))

Disease        Asthma  COVID-19  Cancer  Diabetes  Heart Disease  Hypertension
Disease                                                                       
Asthma             53        25      53        27             31            29
COVID-19           25        47      47        25             29            26
Cancer             53        47     100        48             52            55
Diabetes           27        25      48        48             32            30
Heart Disease      31        29      52        32             52            31
Hypertension       29        26      55        30             31            55

Disease        Asthma  COVID-19  Cancer  Diabetes  Heart Disease  Hypertension
Disease                                                                       
Asthma             50        24      50        24             30            28
COVID-19           24        47      47        25             28            26
Cancer             50        47      97        46  

In [19]:
# (g) Test result patterns: share of positive results per disease
g_o = orig.groupby("Disease")["Result"].apply(lambda s: (s == "Positive").mean())
g_a = final.groupby("Disease")["Result"].apply(lambda s: (s == "Positive").mean())
print(pct(pd.concat([g_o, g_a], axis=1, keys=["original % positive", "anonymized % positive"])))
print("\nOverall % positive:", round((orig["Result"] == "Positive").mean() * 100, 1),
      "(original) vs", round((final["Result"] == "Positive").mean() * 100, 1), "(anonymized)")

               original % positive  anonymized % positive
Disease                                                  
Asthma                        58.5                   60.0
COVID-19                      44.7                   44.7
Cancer                        49.7                   50.6
Diabetes                      43.8                   43.8
Heart Disease                 40.4                   39.2
Hypertension                  41.8                   41.8

Overall % positive: 47.3 (original) vs 47.7 (anonymized)


In [20]:
# (h) Time elapsed between tests for the same person
s = orig.sort_values(["Name", "Test_Date"])
gaps = s.groupby("Name")["Test_Date"].diff().dt.days.dropna()
print("Original: %d gaps, mean %.1f days, median %.1f days, range %d to %d" %
      (len(gaps), gaps.mean(), gaps.median(), gaps.min(), gaps.max()))

# In the anonymized data only the period is left, so we can only compare how tests are spread over the 2 periods
print("People with tests in both periods: original", (orig.groupby("Name")["Period"].nunique() == 2).sum(),
      "| anonymized", (final.groupby("Person")["Period"].nunique() == 2).sum())

Original: 314 gaps, mean 332.5 days, median 241.5 days, range 2 to 1514
People with tests in both periods: original 86 | anonymized 79


# Step 3 (extra checks): how many different people are in each equivalence class

In [21]:
# k by distinct people, class by class (the pseudonym keeps a person's rows linkable)
people_per_class = final.groupby(QI_ANON)["Person"].nunique()
print(people_per_class.value_counts().sort_index())
print("Classes:", len(people_per_class), "| class sizes (rows): min", final.groupby(QI_ANON).size().min(),
      "max", final.groupby(QI_ANON).size().max())

Person
1     1
3     5
4    13
5     1
6     7
7     3
8     2
9     3
Name: count, dtype: int64
Classes: 35 | class sizes (rows): min 5 max 34


# Step 3 (continued): ρ-closeness

In [23]:
def closeness(d, qi, sa="Disease"):
    Q = d[sa].value_counts(normalize=True)                                  # overall distribution of the sensitive attribute
    def dist(s):
        p = s.value_counts(normalize=True).reindex(Q.index, fill_value=0)   # distribution inside one class
        return 0.5 * (p - Q).abs().sum()                                    # total variation distance (= EMD for categorical values)
    return d.groupby(qi)[sa].apply(dist)                                    # one distance per equivalence class

In [24]:
D = closeness(final, QI_ANON, "Disease")
print("rho-closeness (Disease), worst class: %.3f" % D.max())
print("Average class distance: %.3f | median: %.3f" % (D.mean(), D.median()))
for t in [0.2, 0.3, 0.4]:
    print("Classes within t = %.1f: %d of %d" % (t, (D <= t).sum(), len(D)))

print("\nWorst class:")
worst = D.idxmax()
print(worst, "->", dict(final.groupby(QI_ANON)["Disease"].agg(lambda s: dict(s.value_counts())).loc[worst]))
print("\nBefore suppression: %.3f" % closeness(anon, QI_ANON, "Disease").max())
print("Result (secondary sensitive attribute): %.3f" % closeness(final, QI_ANON, "Result").max())

rho-closeness (Disease), worst class: 0.412
Average class distance: 0.221 | median: 0.215
Classes within t = 0.2: 15 of 35
Classes within t = 0.3: 28 of 35
Classes within t = 0.4: 34 of 35

Worst class:
('40-59', 'Female', 'ZIP 1-3', '2019-2021') -> {'Hypertension': np.int64(2), 'Diabetes': np.int64(2), 'Cancer': np.int64(2)}

Before suppression: 0.872
Result (secondary sensitive attribute): 0.334
